In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn import FunctionSampler
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC
from xgboost import XGBClassifier

RANDOM_STATE = 42
OUTER_FOLDS, OUTER_REPEATS, INNER_FOLDS = 5, 3, 5
N_JOBS = 2
DATASET = Path("Datasets/Processed/Dataset Dupilumab Baseline.xlsx - Anagrafica.csv")

class MultiColumnBinarizer(BaseEstimator, TransformerMixin):
    # BaseEstimator: provides parameter handling for sklearn Pipelines and GridSearch
    # TransformerMixin: automatically implements the fit_transform() method

    def __init__(self, min_frequency):
        self.min_frequency = min_frequency
        self.binarizers_ = {}
        self.col_names_ = []
        self.frequent_classes_ = {}


    def _parse_series(self, series):
        series = series.fillna("").astype(str) # Missing values become 0
        parsed = []
        for row in series:
            if not row.strip():
                parsed.append([])
            else:
                items = [
                    item.strip().lower()
                    for item in row.split(",")
                    if item.strip()
                ]
                parsed.append(list(set(items)))
        return parsed

    def _map_rare_to_other(self, lists, freq_set):
        # Maps infrequent or unseen categories into 'other'
        return [list({item if item in freq_set else "other" for item in sublist}) for sublist in lists]

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        self.col_names_ = list(X_df.columns)
        self.binarizers_ = {}
        self.frequent_classes_ = {}

        for col in self.col_names_:
            lists = self._parse_series(X_df[col])
            
            # Count occurrences in the training set and keep classes >= min_frequency
            all_items = [item for sublist in lists for item in sublist]
            counts = pd.Series(all_items).value_counts()
            freq_set = set(counts[counts >= self.min_frequency].index)
            self.frequent_classes_[col] = freq_set

            # Map rare classes to 'other' before fitting MultiLabelBinarizer
            mapped_lists = self._map_rare_to_other(lists, freq_set)
            all_classes = sorted(list(freq_set | {"other"}))

            mlb = MultiLabelBinarizer(classes=all_classes, sparse_output=False)
            mlb.fit(mapped_lists)
            self.binarizers_[col] = mlb

        return self
   

    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.col_names_)
        transformed_parts = []
        
        for col in self.col_names_:
            lists = self._parse_series(X_df[col])
            mapped_lists = self._map_rare_to_other(lists, self.frequent_classes_[col])
            transformed_parts.append(self.binarizers_[col].transform(mapped_lists))
            
        return np.hstack(transformed_parts)

    def get_feature_names_out(self, input_features=None):
        feature_names = []
        for col in self.col_names_:
            prefix = col.lower().replace(" ", "_")
            classes = self.binarizers_[col].classes_
            feature_names.extend([f"{prefix}_{c.replace(' ', '_')}" for c in classes])
        return np.array(feature_names)

# Feature cliniche fisse da Test4. I set A/D cambiano solo SNOT/VAS.
continuous_cols = [
    "Peso (kg)",
    "Altezza (cm)",
    "Neutrofili",
    "Linfociti",
    "Monociti",
    "Eosinofili",
    "Basofili",
    "IgE totali",
]


categorical_cols = ['Categoria Professione', 'Sesso', 'Biologico precedente (0 no, 1 sì)', 'Familiarità Atopia', "OCS nell'ultimo anno", 'Intolleranza ai FANS']

multi_categorical_cols = ["Comorbidità", "Quali allergeni positivi"]
ordinal_cols = ["Fumo Baseline (0=no, 1=ex, 2=si)"]

base_discrete = [
    "Età", "Num Comorbidità", "Numero di pregressi interventi ESS",
    "N. allergeni positivi", "Giorni al mese di utilizzo spray", "ACT", "NPS",
]
vas_ridotto = ["Ostruzione nasale", "Rinorrea anteriore", "Rinorrea posteriore", "Starnuti", "QoL"]
feature_sets = {
    "A": ["NCS", "Dolore facciale", "Starnuti"],
    "B": ["Physical Symptoms", "Health and QoL", "Nasal Symptoms", "Otologic Symptoms",
          "Sleep Symptoms", "Emotional Symptoms", "NCS", "Dolore facciale", *vas_ridotto],
    "C": ["Nasal Symptoms", "Otologic Symptoms", "Sleep Symptoms", "Emotional Symptoms", *vas_ridotto],
    "D": ["Physical Symptoms", "Sleep Symptoms", "Emotional Symptoms", *vas_ridotto],
    "E": ["Physical Symptoms", "Rinorrea anteriore", "Rinorrea posteriore", "Starnuti", "QoL"]
}


def load_dataset():
    for root in (Path.cwd(), *Path.cwd().parents):
        if (root / DATASET).is_file():
            return pd.read_csv(root / DATASET)
    raise FileNotFoundError(DATASET)


def clean_dataset(raw, feature_set):
    discrete = base_discrete + feature_sets[feature_set]
    columns = continuous_cols + discrete + categorical_cols + multi_categorical_cols + ordinal_cols
    X = raw.loc[:, columns].copy()
    for col in continuous_cols + discrete + ordinal_cols:
        values = X[col].astype("string").str.strip()
        european = values.str.contains(",", regex=False, na=False)
        values = values.mask(european, values.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
        X[col] = pd.to_numeric(values, errors="coerce").astype(float)
    return X, raw["TARGET"].astype(int), discrete


def smotenc_resample(X, y, fattore, random_state):
    if fattore == 1:
        return X, y
    prefixes = tuple(col.lower().replace(" ", "_") + "_" for col in multi_categorical_cols)
    cat_idx = [i for i, col in enumerate(X.columns)
               if col in categorical_cols or col in ordinal_cols or col.startswith(prefixes)]
    return SMOTENC(
        categorical_features=cat_idx,
        sampling_strategy={label: round(n * fattore) for label, n in Counter(y).items()},
        k_neighbors=5,
        random_state=random_state,
    ).fit_resample(X, y)


def make_preprocessing(discrete, min_frequency, discrete_imputer, native_nan, use_smote):
    encoder = OneHotEncoder(sparse_output=False, drop="if_binary", handle_unknown="ignore")
    cat_imputer = SimpleImputer(strategy="most_frequent")
    if native_nan:
        continuous_step = discrete_step = ordinal_step = "passthrough"
    else:
        continuous_step = make_pipeline(SimpleImputer(strategy="mean"), StandardScaler())
        discrete_step = make_pipeline(discrete_imputer, StandardScaler())
        ordinal_step = make_pipeline(SimpleImputer(strategy="most_frequent"), StandardScaler())
    pre = ColumnTransformer([
        ("num_cont", continuous_step, continuous_cols),
        ("num_discr", discrete_step, discrete),
        ("cat", cat_imputer if use_smote else make_pipeline(cat_imputer, encoder), categorical_cols),
        ("multi_cat", MultiColumnBinarizer(min_frequency=min_frequency), multi_categorical_cols),
        ("ord", ordinal_step, ordinal_cols),
    ], verbose_feature_names_out=False)
    if not use_smote:
        return [("pre", pre)]
    post = ColumnTransformer([("cat", encoder, categorical_cols)], remainder="passthrough")
    return [
        ("pre", pre.set_output(transform="pandas")),
        ("smotenc", FunctionSampler(func=smotenc_resample, validate=False)),
        ("post", post),
    ]


def validate_model(X, y, pipeline, param_grid, factors):
    outer = RepeatedStratifiedKFold(
        n_splits=OUTER_FOLDS, n_repeats=OUTER_REPEATS, random_state=RANDOM_STATE
    )
    labels = np.sort(y.unique())
    rows = []
    matrices = []
    for split_num, (train, test) in enumerate(outer.split(X, y)):
        repeat, fold = divmod(split_num, OUTER_FOLDS)
        seed = RANDOM_STATE + split_num
        estimator = clone(pipeline)
        if "model__random_state" in estimator.get_params():
            estimator.set_params(model__random_state=seed)
        grids = param_grid
        if factors:
            model_grids = param_grid if isinstance(param_grid, list) else [param_grid]
            grids = [
                {**grid, "smotenc__kw_args": [
                    {"fattore": factor, "random_state": seed} for factor in factors
                ]}
                for grid in model_grids
            ]
        inner = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=seed)
        search = GridSearchCV(estimator, grids, cv=inner, scoring="f1_macro",
                              n_jobs=N_JOBS, error_score="raise")
        print(f"Ripetizione {repeat + 1}/{OUTER_REPEATS}, fold {fold + 1}/{OUTER_FOLDS}", flush=True)
        search.fit(X.iloc[train], y.iloc[train])
        actual = y.iloc[test]
        pred = search.predict(X.iloc[test])
        rows.append({
            "ripetizione": repeat + 1, "fold": fold + 1,
            "f1_macro": f1_score(actual, pred, average="macro"),
            "accuracy": accuracy_score(actual, pred),
            "precision": precision_score(actual, pred, zero_division=0),
            "recall": recall_score(actual, pred, zero_division=0),
            "best_params": search.best_params_,
        })
        matrices.append(confusion_matrix(actual, pred, labels=labels))
    return pd.DataFrame(rows), np.sum(matrices, axis=0), labels


def plot_results(name, results, matrix, labels):
    metrics = ["f1_macro", "accuracy", "precision", "recall"]
    summary = results[metrics].agg(["mean", "std"]).T
    display(summary.round(4))
    display(results.drop(columns="best_params").round(4))
    parametri = pd.json_normalize(results["best_params"])
    parametri.columns = parametri.columns.str.replace(r"^model__", "", regex=True)
    parametri.columns = parametri.columns.str.replace(r"^smotenc__kw_args\.", "SMOTENC: ", regex=True)
    print("Migliori parametri per fold")
    with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
        display(pd.concat([results[["ripetizione", "fold"]].reset_index(drop=True), parametri], axis=1))
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    axes[0].bar(summary.index, summary["mean"], yerr=summary["std"], capsize=4)
    axes[0].set(ylim=(0, 1.05), title="Metriche esterne (media e deviazione standard)")
    ConfusionMatrixDisplay(matrix, display_labels=labels).plot(ax=axes[1], colorbar=False)
    axes[1].set_title("Predizioni aggregate su tutte le ripetizioni")
    fig.suptitle(name)
    plt.show()


In [ ]:
# LinearSVC | set C
# Valori da compilare con la configurazione scelta.
linear_c = 0.001
linear_smotenc_factor = 1.8 
linear_seed = 0  # Seed dei fold esterni e di SMOTENC.

# Dataset e pulizia
raw = load_dataset()
X, y, discrete = clean_dataset(raw, "C")

# Preprocessing e modello
steps = make_preprocessing(
    discrete, min_frequency=2, discrete_imputer=SimpleImputer(strategy='median'),
    native_nan=False, use_smote=True,
)
# La formulazione primale con squared hinge evita il solver duale di hinge.
model = LinearSVC(C=linear_c, loss='squared_hinge', dual=False, tol=1e-3, max_iter=100_000, class_weight='balanced')
smotenc_args = {"fattore": linear_smotenc_factor, "random_state": linear_seed}
pipeline = ImbPipeline([*steps, ("model", model)]).set_params(smotenc__kw_args=smotenc_args)

# Validazione e grafici
outer = RepeatedStratifiedKFold(
    n_splits=OUTER_FOLDS, n_repeats=OUTER_REPEATS, random_state=linear_seed
)
labels = np.sort(y.unique())
rows = []
matrix = np.zeros((len(labels), len(labels)), dtype=int)
for split_num, (train, test) in enumerate(outer.split(X, y)):
    repeat, fold = divmod(split_num, OUTER_FOLDS)
    fitted = clone(pipeline).fit(X.iloc[train], y.iloc[train])
    actual = y.iloc[test]
    pred = fitted.predict(X.iloc[test])
    rows.append({
        "ripetizione": repeat + 1, "fold": fold + 1,
        "f1_macro": f1_score(actual, pred, average="macro"),
        "accuracy": accuracy_score(actual, pred),
        "precision": precision_score(actual, pred, zero_division=0),
        "recall": recall_score(actual, pred, zero_division=0),
        "best_params": {"model__C": linear_c, "smotenc__kw_args": smotenc_args},
    })
    matrix += confusion_matrix(actual, pred, labels=labels)
results_linear_svc = pd.DataFrame(rows)
plot_results("LinearSVC", results_linear_svc, matrix, labels)


In [ ]:
# LogisticRegression | set E
# Dataset e pulizia
raw = load_dataset()
X, y, discrete = clean_dataset(raw, "E")

# Preprocessing e modello
factors = (1.8)
steps = make_preprocessing(
    discrete, min_frequency=2, discrete_imputer=SimpleImputer(strategy="median"),
    native_nan=False, use_smote=bool(factors),
)
model = LogisticRegression(max_iter=10_000, tol=0.001, random_state=RANDOM_STATE, class_weight='balanced')
pipeline = (ImbPipeline if factors else Pipeline)([*steps, ("model", model)])
param_grid = [{'model__C': [0.1], 'model__solver': ['saga'], 'model__l1_ratio': [0.5]}]

# Validazione e grafici
results_log_reg, matrix, labels = validate_model(X, y, pipeline, param_grid, factors)
plot_results("LogisticRegression", results_log_reg, matrix, labels)


In [ ]:
# RBFSVC | set A
# Dataset e pulizia
raw = load_dataset()
X, y, discrete = clean_dataset(raw, "A")

# Preprocessing e modello
factors = (1.0, 1.5, 1.8)
steps = make_preprocessing(
    discrete, min_frequency=4, discrete_imputer=SimpleImputer(strategy="median"),
    native_nan=False, use_smote=bool(factors),
)
model = SVC(kernel='rbf', class_weight='balanced')
pipeline = (ImbPipeline if factors else Pipeline)([*steps, ("model", model)])
param_grid = {'model__C': [0.1, 1, 10], 'model__gamma': ['scale', 0.01, 0.1, 1]}

# Validazione e grafici
results_rbf_svc, matrix, labels = validate_model(X, y, pipeline, param_grid, factors)
plot_results("RBFSVC", results_rbf_svc, matrix, labels)


In [ ]:
# KNeighborsClassifier | set A
# Dataset e pulizia
raw = load_dataset()
X, y, discrete = clean_dataset(raw, "A")

# Preprocessing e modello
factors = ()
steps = make_preprocessing(
    discrete, min_frequency=4, discrete_imputer=SimpleImputer(strategy="median"),
    native_nan=False, use_smote=bool(factors),
)
model = KNeighborsClassifier()
pipeline = (ImbPipeline if factors else Pipeline)([*steps, ("model", model)])
param_grid = {'model__n_neighbors': [3, 5, 7, 11], 'model__weights': ['uniform', 'distance'], 'model__p': [1, 2]}

# Validazione e grafici
results_knn, matrix, labels = validate_model(X, y, pipeline, param_grid, factors)
plot_results("KNeighborsClassifier", results_knn, matrix, labels)


In [ ]:
# RandomForestClassifier | set B
# Dataset e pulizia
raw = load_dataset()
X, y, discrete = clean_dataset(raw, "B")

# Preprocessing e modello
factors = (1.5, 1.8)
steps = make_preprocessing(
    discrete, min_frequency=2, discrete_imputer=SimpleImputer(strategy="median"),
    native_nan=False, use_smote=bool(factors),
)
model = RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE)
pipeline = (ImbPipeline if factors else Pipeline)([*steps, ("model", model)])
param_grid = {'model__n_estimators': [50, 100, 500], 'model__max_depth': [None, 5, 10, 50], 'model__min_samples_leaf': [1, 2, 5, 10], 'model__max_features': [None, 'sqrt', 'log2']}

# Validazione e grafici
results_rf, matrix, labels = validate_model(X, y, pipeline, param_grid, factors)
plot_results("RandomForestClassifier", results_rf, matrix, labels)


In [ ]:
# XGBClassifier | set D
# Dataset e pulizia
raw = load_dataset()
X, y, discrete = clean_dataset(raw, "D")

# Preprocessing e modello
factors = ()
steps = make_preprocessing(
    discrete, min_frequency=2, discrete_imputer=None,
    native_nan=True, use_smote=bool(factors),
)
model = XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=1)
pipeline = (ImbPipeline if factors else Pipeline)([*steps, ("model", model)])
param_grid = {'model__n_estimators': [50, 100], 'model__max_depth': [2, 3], 'model__learning_rate': [0.05, 0.1], 'model__min_child_weight': [1, 3, 5]}

# Validazione e grafici
results_xgb, matrix, labels = validate_model(X, y, pipeline, param_grid, factors)
plot_results("XGBClassifier", results_xgb, matrix, labels)
